## 1. Imports

In [ ]:
import os
from itertools import islice

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42

## 2. Load 120,000 reviews

In [ ]:
if os.path.exists("yelp_120k.csv"):
    df = pd.read_csv("yelp_120k.csv")
else:
    from datasets import load_dataset
    stream = load_dataset("Yelp/yelp_review_full", split="train", streaming=True)
    stream = stream.shuffle(seed=SEED, buffer_size=10000)
    df = pd.DataFrame(list(islice(stream, 120000)))
    df.to_csv("yelp_120k.csv", index=False)

df = df.dropna().drop_duplicates(subset="text")
df["n_words"] = df["text"].str.split().str.len()
df.head()

## 3. Keep reviews with 40-200 words

In [ ]:
df = df[df["n_words"].between(40, 200)]
df["label"].value_counts().sort_index()

## 4. Split: 30,000 train / 10,000 validation / 10,000 test

In [ ]:
per_class = min(10000, df["label"].value_counts().min())
data = df.groupby("label").sample(n=per_class, random_state=SEED)

train, temp = train_test_split(data, test_size=20000, stratify=data["label"], random_state=SEED)
val, test = train_test_split(temp, test_size=10000, stratify=temp["label"], random_state=SEED)

train[["text", "label"]].to_csv("train.csv", index=False)
val[["text", "label"]].to_csv("val.csv", index=False)
test[["text", "label"]].to_csv("test.csv", index=False)

print(f"Train: {len(train):,} | Validation: {len(val):,} | Test: {len(test):,} | Classes: {data['label'].nunique()}")

## 5. Word count distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(data["n_words"], bins=range(40, 205, 5), color="#2a78d6", edgecolor="white")
ax.axvline(data["n_words"].median(), color="black", linestyle="--",
           label=f"median = {data['n_words'].median():.0f} words")
ax.set_title("Word count per review (40-200 words)", loc="left", fontweight="bold")
ax.set_xlabel("Words per review")
ax.set_ylabel("Reviews")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()